In [0]:
# import libraries

import boto3
import pandas as pd
import re

In [0]:
# connect to S3 and find assessment files

s3 = boto3.client('s3')

paginator = s3.get_paginator("list_objects_v2")

bucket_name = "data602-final-project"

assessment_files = [
    obj["Key"]
    for page in paginator.paginate(Bucket=bucket_name)
    for obj in page.get("Contents", [])
    if obj["Key"].startswith("Talent/")
    and obj["Key"].endswith(".txt")
]

In [0]:
# clean and transform assessment data

rows = []

for file in assessment_files:
    obj = s3.get_object(Bucket=bucket_name, Key=file)
    body = obj["Body"].read().decode("utf-8")
    lines = body.splitlines()

    assessment_date = None
    academy_location = None

    for line in lines:
        line = line.strip()

        if line == "":
            continue

        if re.search(r"\w+ \d{1,2} \w+ \d{4}", line):
            assessment_date = line
            continue

        if "Academy" in line:
            academy_location = line
            continue

        if "Psychometrics:" in line and "Presentation:" in line:
            match = re.match(
                r"(.+?)\s+-\s+Psychometrics:\s+(\d+)/100,\s+Presentation:\s+(\d+)/32",
                line
            )

            if match:
                rows.append({
                    "candidate_name": match.group(1).title(),
                    "assessment_date": assessment_date,
                    "location": academy_location,
                    "psychometric_score": int(match.group(2)),
                    "presentation_score": int(match.group(3))
                })

In [0]:
# create silver assessments table

silver_assessments = pd.DataFrame(rows)

spark_df = spark.createDataFrame(silver_assessments)

spark_df.write.mode("overwrite").saveAsTable("silver_assessments")

display(spark.table("silver_assessments"))

candidate_name,assessment_date,location,psychometric_score,presentation_score
Phyllys Baelde,Thursday 15 August 2019,Birmingham Academy,69,24
Cecile Lates,Thursday 15 August 2019,Birmingham Academy,55,22
Charlean Devons,Thursday 15 August 2019,Birmingham Academy,52,14
Avrit Gawith,Thursday 15 August 2019,Birmingham Academy,59,23
Doralin Purkess,Thursday 15 August 2019,Birmingham Academy,49,24
Nikolos Yashin,Thursday 15 August 2019,Birmingham Academy,57,20
Emmanuel Deniske,Thursday 15 August 2019,Birmingham Academy,50,16
Mella Aubin,Thursday 15 August 2019,Birmingham Academy,51,22
Cal Loache,Thursday 15 August 2019,Birmingham Academy,53,20
Salaidh Loveday,Thursday 15 August 2019,Birmingham Academy,48,16
